# 04 — MICM Fuel Prices | Automatic Download (raw landing)

Downloads the MICM weekly fuel-price files published on the national open-data portal and lands them, unchanged, in the raw archive
`/Volumes/rd_economic_intelligence/bronze/micm_raw/fuel_prices/`.

| Step | Rule |
|---|---|
| Discover | datos.gob.do CKAN API, dataset `precios-de-los-combustibles`: current URL of the **XLSX** (authoritative) and **CSV** (reconciliation) resources. The file name carries the year range (`…-2010-2026`), so the URL is read from the portal on every run instead of being hard-coded. |
| Validate | XLSX must contain the columns `DIA DESDE`, `DIA HASTA`, `MES`, `AÑO` and the main fuels; the CSV header must contain `DIA DESDE`. Otherwise the task fails and nothing is written. |
| Land | Shared helper `src/shared/acquisition.py`: identical content is never landed twice; new content goes to `<yyyy>/<mm>/`; a republished file with an archived name is kept as `<name>__<date>`. |
| Source down | If the site cannot be reached after 3 attempts (network, invalid TLS certificate, HTTP 5xx), the task ends as a warning (`SOURCE_UNAVAILABLE`), the outage is logged in `quality.source_availability` and the source keeps its last data. HTTP 4xx and layout changes still fail. |
| Signal | Task value `downloaded` = true when at least one new file landed. |

Bronze (`04_MICM_Fuel_Prices_Bronze`) then captures new files incrementally with Auto Loader. **Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, io, json, importlib
import pandas as pd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
importlib.reload(ACQ)

LANDING = "/Volumes/rd_economic_intelligence/bronze/micm_raw/fuel_prices/"
PACKAGE = "precios-de-los-combustibles"
REQUIRED = ["DIA DESDE", "DIA HASTA", "MES", "AÑO", "GASOLINA PREMIUM", "GASOLINA REGULAR", "GASOIL OPTIMO", "GLP"]

try:
    urls, portal_modified = ACQ.ckan_resources(PACKAGE, ["xlsx", "csv"])
    print(f"Portal dataset updated {portal_modified}: {urls}")

    files = {fmt: ACQ.http_get(url) for fmt, url in urls.items()}
    cols = [str(c).strip() for c in pd.read_excel(io.BytesIO(files["xlsx"]), dtype=object, nrows=5).columns]
    missing = [c for c in REQUIRED if c not in cols]
    if missing:
        raise ValueError(f"XLSX failed the structural contract, missing columns: {missing}")
    if "DIA DESDE" not in files["csv"][:500].decode("latin-1"):
        raise ValueError("CSV failed the structural contract: header 'DIA DESDE' not found")

    results = {fmt: ACQ.land(data, ACQ.urllib.parse.unquote(urls[fmt].rsplit("/", 1)[-1]), LANDING) for fmt, data in files.items()}
    downloaded = any(r["status"] == "DOWNLOADED" for r in results.values())
    dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)
    dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED" if downloaded else "SKIPPED", "portal_modified": portal_modified, "files": results}, ensure_ascii=False))
except ACQ.SourceUnavailable as e:
    ACQ.source_unavailable(dbutils, spark, "micm_fuel_prices", e)